# 第11課 - 代理人對代理人 (A2A) 協議


## 設定


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 什麼是 A2A 協定？

The **Agent-to-Agent (A2A) protocol** 是一個開放標準，使 AI 代理人能夠溝通、彼此發現並協作 — 即使它們建立在不同的框架上或由不同的服務託管。

Key concepts:

- **Discovery** – 代理人會發布一張 *代理人卡片*，描述其能力，讓其他代理人（或協調者）能輕鬆找到適合該任務的專家。
- **Message Passing** – 代理人透過共同的協定交換結構化訊息，所以一個代理人的請求可以被另一個代理人理解並完成，無論其內部實作為何。
- **Task Lifecycle** – A2A 定義像 *submitted*、*working*、*completed* 和 *failed* 等狀態，讓協調者能完整掌握被委派任務的進度。

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents into a workflow where each agent contributes its expertise and passes results to the next.


## 建立專門的旅遊代理人


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## 透過工作流程的多代理人協作

我們將這三個代理人連接成一個順序的工作流程，以模擬 A2A 訊息傳遞：

1. **CurrencyExchangeAgent** 接收使用者的請求並產生貨幣指引。
2. **ActivityPlannerAgent** 接收增強後的上下文並加入活動建議。
3. **TravelManagerAgent** 將兩者輸入綜合為最終的旅遊簡報。


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## 在生產環境中理解 A2A

在生產環境中，A2A 協議可解鎖強大的跨服務情境：

| Capability | Description |
|---|---|
| **Cross-framework interop** | 使用一個框架建構的代理可以將任務委派給由任何其他符合 A2A 的框架建構的代理，實現真正的跨組織互通。 |
| **Service boundaries** | 代理可以存在於不同的微服務、雲端區域，或甚至不同的組織，同時仍能無縫協作。 |
| **Dynamic discovery** | 協調者可以在執行時查詢 Agent Card 註冊表，以找到最適合特定子任務的專家。 |
| **Streaming & push notifications** | A2A 支援 Server-Sent Events (SSE) 以進行即時進度更新，以及用於長時間執行任務的推播通知。 |

我們上面建立的工作流程是此模式的一個簡化、進程內版本。  
在實際的  
部署中，每個代理會公開一個 HTTP 端點、發佈 Agent Card，並透過  
A2A JSON-RPC 協議進行通訊。


## 摘要

在本課程中您學到：

1. **A2A 協議是什麼** — 一個用於代理對代理發現、訊息傳遞、  
   與任務管理。
2. **如何建立專門的代理** — 一個貨幣兌換代理、一個活動規劃代理、  
   以及一個旅遊管理協調者。
3. **如何將代理串接到工作流程中** — 使用 `WorkflowBuilder` 來模擬代理之間的順序  
   訊息傳遞。
4. **A2A 在生產環境中的運作方式** — 使得跨框架、跨服務的協作成為可能，  
   並支援動態發現與串流更新。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
免責聲明：
本文件使用 AI 翻譯服務「Co‑op Translator」(https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意自動翻譯可能包含錯誤或不精確之處。原始語言的文件應視為權威版本。對於重要資訊，建議採用專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
